In [1]:
from pathlib import Path
import pandas as pd
from rdkit.Chem import PandasTools, Descriptors, rdMolDescriptors
from sklearn.feature_selection import VarianceThreshold
from sklearn.ensemble import RandomForestRegressor
import numpy as np

desc_list = Descriptors.descList
desc_name = [name for name, func in desc_list]
desc_func = [func for name, func in desc_list]

def mol_to_allfeatures(mol):
    return [func(mol) for func in desc_func] + rdMolDescriptors.MQNs_(mol)

file = Path.cwd().parent / "data" / "test.xlsx"
df = pd.read_excel(file)
PandasTools.AddMoleculeColumnToFrame(df, "SMILES", "ROMol", False)

data = df["ROMol"].apply(mol_to_allfeatures).tolist()
for i in range(42):
    desc_name += ["mqn" + str(i + 1)]
df1 = pd.DataFrame(data, columns = desc_name)

var_selector = VarianceThreshold(threshold = 5)
df1_var = var_selector.fit_transform(df1)
selected_columns = df1.columns[var_selector.get_support()]
df2 = pd.DataFrame(df1_var, columns = selected_columns)

corr = df2.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k = 1).astype(bool))
to_drop = [col for col in upper.columns if any(upper[col] > 0.9)]
df2 = df2.drop(columns = to_drop)

Importances = RandomForestRegressor().fit(df2.values, df["Q(cal/g)"]).feature_importances_
feature_importances = pd.Series(Importances, index = df2.columns)
print(feature_importances.sort_values(ascending = False))

fr_NH0         0.410305
VSA_EState3    0.155885
EState_VSA8    0.036377
VSA_EState4    0.031118
PEOE_VSA7      0.030780
PEOE_VSA8      0.023100
PEOE_VSA1      0.023012
SlogP_VSA4     0.021929
BertzCT        0.018273
PEOE_VSA3      0.016225
PEOE_VSA2      0.013820
fr_Ar_N        0.013576
VSA_EState2    0.010672
SMR_VSA3       0.010509
SlogP_VSA1     0.009493
PEOE_VSA14     0.008919
TPSA           0.008428
PEOE_VSA5      0.007340
SlogP_VSA8     0.007246
VSA_EState1    0.007230
EState_VSA9    0.006620
SlogP_VSA2     0.006611
PEOE_VSA12     0.006600
PEOE_VSA10     0.006497
EState_VSA5    0.005912
SMR_VSA1       0.005818
VSA_EState6    0.005709
PEOE_VSA13     0.005517
SMR_VSA10      0.005194
mqn1           0.005035
SMR_VSA9       0.004970
PEOE_VSA6      0.004595
SMR_VSA4       0.004465
EState_VSA1    0.004209
VSA_EState5    0.004021
SPS            0.003933
EState_VSA3    0.003906
EState_VSA2    0.003754
PEOE_VSA11     0.003691
SlogP_VSA10    0.003283
SMR_VSA5       0.003255
PEOE_VSA9      0